# Costs, calibration, and held-out routing

Demonstration documents only. These notebooks inspect the committed evidence; they do not claim a separate full-dataset training run.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['scope'])


Population: {'generated': {'train': 48, 'validation': 24, 'test': 24}, 'cord': {'train': 800, 'validation': 100, 'test': 100}, 'limitations': ['Full original CORD 800/100/100 splits are used, with a project-specific subtotal/tax/total and line-item mapping, not the official benchmark metric.', 'Generated invoices share four templates; performance does not establish generalization to unseen vendors or layouts.', 'Small test populations give wide intervals; the chosen costs are illustrative (review 1, wrong approval 20).', 'CORD gold words cover annotated semantic lines; full-page OCR CER/WER includes unannotated text.', 'Gold/real OCR input gaps include token segmentation and layout conventions, so they are not pure additive character-recognition loss.', 'Rules are conservative on receipts without invoice date, vendor and PO evidence, so those receipts go to review.', 'Other-class classification uses generated text tokens; no real miscellaneous mailroom documents were evaluated.', 'Extr

In [2]:
op = e['operating_point']
print({key:value for key,value in op.items() if key != 'curve'})
print(pd.DataFrame(op['curve']).to_string(index=False))
calibration = [{key:value for key,value in row.items() if key != 'bins'} for row in e['calibration']]
print(pd.DataFrame(calibration).to_string(index=False))
assert op['chosen_on'] == 'validation'
assert op['reported_on'] == 'test'

{'threshold': 0.9, 'chosen_on': 'validation', 'review_cost': 1, 'error_cost': 20, 'validation_n': 122, 'unlabeled_validation_documents': 2, 'reported_on': 'test', 'backend': 'rules', 'ocr_engine': 'tesseract', 'n': 124, 'auto_approved': 12, 'review_rate': {'value': 0.9032258064516129, 'ci': [0.838447066252923, 0.9437717923433059], 'n': 124, 'method': '95% Wilson score interval'}, 'auto_approval_error': {'value': 0.0, 'ci': [0, 0.24249400665523077], 'n': 12, 'method': '95% Wilson score interval'}, 'unlabeled_auto_approved': 0, 'unlabeled_test_documents': 5, 'correctness': 'All required normalized fields correct and no planted business defect', 'by_source': [{'source': 'generated', 'n': 24, 'auto_approved': 12, 'review_rate': {'value': 0.5, 'ci': [0.31427425819573795, 0.685725741804262], 'n': 24, 'method': '95% Wilson score interval'}, 'unlabeled_documents': 0, 'unlabeled_auto_approved': 0, 'auto_approval_error': {'value': 0.0, 'ci': [0, 0.24249400665523077], 'n': 12, 'method': '95% Wils

## Rejected shortcut / negative control

In [3]:
# Negative control: test-selected routing must be rejected by publication.
from copy import deepcopy
from scripts.render_reports import validate_manifest
invalid = deepcopy(e)
invalid['operating_point']['chosen_on'] = 'test'
try:
    validate_manifest(invalid)
except ValueError as error:
    print('Rejected:', error)
else:
    raise AssertionError('Test leakage was not rejected')

Rejected: Routing must be selected on validation and reported on test
